In [1]:
import pandas as pd

books = pd.read_csv("books_with_categories.csv")

In [3]:
from transformers import pipeline
classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base",top_k=None,device="mps")
classifier("I love this!")

Device set to use mps


[[{'label': 'joy', 'score': 0.9771687984466553},
  {'label': 'surprise', 'score': 0.008528691716492176},
  {'label': 'neutral', 'score': 0.005764603149145842},
  {'label': 'anger', 'score': 0.004419785924255848},
  {'label': 'sadness', 'score': 0.002092392183840275},
  {'label': 'disgust', 'score': 0.0016119939973577857},
  {'label': 'fear', 'score': 0.0004138523945584893}]]

In [4]:
sentences = books["description"][0].split(".")
predictions = classifier(sentences)
predictions[0]

[{'label': 'surprise', 'score': 0.7296028733253479},
 {'label': 'neutral', 'score': 0.14038559794425964},
 {'label': 'fear', 'score': 0.06816215068101883},
 {'label': 'joy', 'score': 0.047942452132701874},
 {'label': 'anger', 'score': 0.00915634073317051},
 {'label': 'disgust', 'score': 0.0026284719351679087},
 {'label': 'sadness', 'score': 0.0021221598144620657}]

In [5]:
predictions

[[{'label': 'surprise', 'score': 0.7296028733253479},
  {'label': 'neutral', 'score': 0.14038559794425964},
  {'label': 'fear', 'score': 0.06816215068101883},
  {'label': 'joy', 'score': 0.047942452132701874},
  {'label': 'anger', 'score': 0.00915634073317051},
  {'label': 'disgust', 'score': 0.0026284719351679087},
  {'label': 'sadness', 'score': 0.0021221598144620657}],
 [{'label': 'neutral', 'score': 0.44937148690223694},
  {'label': 'disgust', 'score': 0.2735908031463623},
  {'label': 'joy', 'score': 0.10908294469118118},
  {'label': 'sadness', 'score': 0.09362732619047165},
  {'label': 'anger', 'score': 0.040478236973285675},
  {'label': 'surprise', 'score': 0.026970215141773224},
  {'label': 'fear', 'score': 0.006879065651446581}],
 [{'label': 'neutral', 'score': 0.6462157964706421},
  {'label': 'sadness', 'score': 0.24273362755775452},
  {'label': 'disgust', 'score': 0.04342261329293251},
  {'label': 'surprise', 'score': 0.028300536796450615},
  {'label': 'joy', 'score': 0.01421

In [14]:
sorted(predictions[0],key = lambda x:x["label"])

[{'label': 'anger', 'score': 0.01765841245651245},
 {'label': 'disgust', 'score': 0.17792744934558868},
 {'label': 'fear', 'score': 0.04945699870586395},
 {'label': 'joy', 'score': 0.03219756484031677},
 {'label': 'neutral', 'score': 0.6624059081077576},
 {'label': 'sadness', 'score': 0.013540072366595268},
 {'label': 'surprise', 'score': 0.04681359604001045}]

In [24]:
import numpy as np
emotion_labels = ["anger" , "disgust" , "fear","joy","sadness","surprise" ,"neutral"]
isbn = []
emotion_scores = {label:[] for label in emotion_labels}

def calculate_max_emotion_scores(predictions):
    per_emotion_scores={label:[] for label in emotion_labels}
    for prediction in predictions:
        sorted_predictions = sorted(prediction,key = lambda x:x["label"])
        for index , label in enumerate(emotion_labels):
            per_emotion_scores[label].append(sorted_predictions[index]["score"])
    return {
        label: float(np.max(scores)) if scores else 0.0
        for label, scores in per_emotion_scores.items()
    }

In [25]:
for i in range(10):
    desc = books['description'][i]

    # Check if description exists and isn't just whitespace
    if pd.isna(desc) or str(desc).strip() == "":
        print(f"Skipping book {i}: No description found.")
        continue

    isbn.append(books["isbn13"][i])

    # Split and filter out any empty strings
    sentences = [s.strip() for s in str(desc).split(".") if s.strip()]

    # Safety check: if no valid sentences were found after filtering
    if not sentences:
        print(f"Skipping book {i}: No valid sentences after split.")
        continue

    predictions = classifier(sentences)

    # Only calculate scores if we actually got predictions back
    if predictions:
        max_scores = calculate_max_emotion_scores(predictions)
        for label in emotion_labels:
            emotion_scores[label].append(max_scores[label])
    else:
        print(f"Skipping book {i}: Classifier returned no predictions.")

In [26]:
emotion_scores

{'anger': [0.029641296714544296,
  0.5944691896438599,
  0.04130084440112114,
  0.3253042697906494,
  0.09173259884119034,
  0.20615476369857788,
  0.5513842701911926,
  0.02365623600780964,
  0.30067047476768494,
  0.01765841245651245],
 'disgust': [0.33823806047439575,
  0.46199119091033936,
  0.02456834353506565,
  0.12576301395893097,
  0.19743406772613525,
  0.757377564907074,
  0.18220360577106476,
  0.056707713752985,
  0.2874763309955597,
  0.17792744934558868],
 'fear': [0.9839729070663452,
  0.9352148771286011,
  0.9732851982116699,
  0.43633902072906494,
  0.09504326432943344,
  0.03679509088397026,
  0.7008533477783203,
  0.4044956862926483,
  0.8970003128051758,
  0.04945699870586395],
 'joy': [0.949027419090271,
  0.70442134141922,
  0.7672376036643982,
  0.24220913648605347,
  0.041145846247673035,
  0.04337577894330025,
  0.8725654482841492,
  0.013375181704759598,
  0.01669342815876007,
  0.03219756484031677],
 'sadness': [0.697845995426178,
  0.8911097049713135,
  0.0

In [27]:
from tqdm import tqdm

emotion_labels = ["anger" , "disgust" , "fear","joy","sadness","surprise" ,"neutral"]
isbn = []
emotion_scores = {label:[] for label in emotion_labels}

for i in tqdm(range(len(books))):
    desc = books['description'][i]

    # Check if description exists and isn't just whitespace
    if pd.isna(desc) or str(desc).strip() == "":
        print(f"Skipping book {i}: No description found.")
        continue

    isbn.append(books["isbn13"][i])

    # Split and filter out any empty strings
    sentences = [s.strip() for s in str(desc).split(".") if s.strip()]

    # Safety check: if no valid sentences were found after filtering
    if not sentences:
        print(f"Skipping book {i}: No valid sentences after split.")
        continue

    predictions = classifier(sentences)

    # Only calculate scores if we actually got predictions back
    if predictions:
        max_scores = calculate_max_emotion_scores(predictions)
        for label in emotion_labels:
            emotion_scores[label].append(max_scores[label])
    else:
        print(f"Skipping book {i}: Classifier returned no predictions.")

100%|██████████| 5230/5230 [04:17<00:00, 20.31it/s]


In [29]:
emotions_df = pd.DataFrame(emotion_scores)
emotions_df["isbn13"] = isbn

In [30]:
emotions_df.head()

,anger,disgust,fear,joy,sadness,surprise,neutral,isbn13
0,0.029641,0.338238,0.983973,0.949027,0.697846,0.956065,0.729603,9780002005883
1,0.594469,0.461991,0.935215,0.704421,0.891110,0.051414,0.212222,9780002261982
2,0.041301,0.024568,0.973285,0.767238,0.042176,0.010860,0.009796,9780006178736
3,0.325304,0.125763,0.436339,0.242209,0.732687,0.043272,0.029084,9780006280897
4,0.091733,0.197434,0.095043,0.041146,0.890048,0.475881,0.074878,9780006280934


In [31]:
books = pd.merge(books,emotions_df,on="isbn13")

In [33]:
books

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,...,title_and_subtitle,tagged_description,simple_categories,anger,disgust,fear,joy,sadness,surprise,neutral
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,...,Gilead,9780002005883 A NOVEL THAT READERS and critics...,Fiction,0.029641,0.338238,0.983973,0.949027,0.697846,0.956065,0.729603
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,...,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...,Fiction,0.594469,0.461991,0.935215,0.704421,0.891110,0.051414,0.212222
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,...,Rage of angels,"9780006178736 A memorable, mesmerizing heroine...",Fiction,0.041301,0.024568,0.973285,0.767238,0.042176,0.010860,0.009796
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,...,The Four Loves,9780006280897 Lewis' work on the nature of lov...,Nonfiction,0.325304,0.125763,0.436339,0.242209,0.732687,0.043272,0.029084
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,...,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le...",Nonfiction,0.091733,0.197434,0.095043,0.041146,0.890048,0.475881,0.074878
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5225,9788172235222,8172235224,Mistaken Identity,Nayantara Sahgal,Indic fiction (English),http://books.google.com/books/content?id=q-tKP...,On A Train Journey Home To North India After L...,2003.0,2.93,324.0,...,Mistaken Identity,9788172235222 On A Train Journey Home To North...,Fiction,0.204143,0.053939,0.923713,0.303276,0.807058,0.974265,0.028640
5226,9788173031014,8173031010,Journey to the East,Hermann Hesse,Adventure stories,http://books.google.com/books/content?id=rq6JP...,This book tells the tale of a man who goes on ...,2002.0,3.70,175.0,...,Journey to the East,9788173031014 This book tells the tale of a ma...,Nonfiction,0.058449,0.127941,0.025688,0.400263,0.891073,0.016014,0.227764
5227,9788179921623,817992162X,The Monk Who Sold His Ferrari: A Fable About F...,Robin Sharma,Health & Fitness,http://books.google.com/books/content?id=c_7mf...,"Wisdom to Create a Life of Passion, Purpose, a...",2003.0,3.82,198.0,...,The Monk Who Sold His Ferrari: A Fable About F...,9788179921623 Wisdom to Create a Life of Passi...,Fiction,0.011246,0.010868,0.314811,0.942169,0.344739,0.060436,0.056820
5228,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,...,I Am that: Talks with Sri Nisargadatta Maharaj,9788185300535 This collection of the timeless ...,Nonfiction,0.034505,0.080505,0.409872,0.776052,0.950763,0.317456,0.049227


In [34]:
books.to_csv("books_with_emotions.csv",index=False)